# ML-Based Short-Term Futures Price Prediction Using Order Book Data

This notebook implements a machine learning pipeline to predict short-term futures prices using real-time order book data fetched via WebSocket connections.

## 1. Import Required Libraries

Import libraries for data fetching, processing, and modeling.

### GPU Availability Check

In [ ]:
# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPUs available: {len(gpus)}")
    for gpu in gpus:
        print(f"GPU: {gpu}")
else:
    print("No GPUs available. Using CPU.")

In [1]:
import websocket
import json
import pandas as pd
import numpy as np
from datetime import datetime
import time
import threading
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Conv2D, Flatten
import matplotlib.pyplot as plt

2025-11-14 16:08:46.944714: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-14 16:08:47.369817: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-14 16:08:48.718758: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## 2. Establish WebSocket Connection for Order Book Data

Set up a WebSocket connection to Binance Futures API to stream real-time order book data.

In [2]:
# WebSocket URL for Binance Futures Order Book
SYMBOL = 'btcusdt'
DEPTH_LEVELS = 20
UPDATE_SPEED = 100  # ms

ws_url = f'wss://fstream.binance.com/ws/{SYMBOL}@depth{DEPTH_LEVELS}@{UPDATE_SPEED}ms'

# Global variables to store data
order_book_data = []
data_lock = threading.Lock()

def on_message(ws, message):
    data = json.loads(message)
    with data_lock:
        order_book_data.append({
            'timestamp': datetime.now(),
            'bids': data['b'],
            'asks': data['a'],
            'lastUpdateId': data['u']
        })

def on_error(ws, error):
    print(f"Error: {error}")

def on_close(ws, close_status_code, close_msg):
    print("Connection closed")

def on_open(ws):
    print("Connection opened")

# Function to start WebSocket in a thread
def start_websocket():
    ws = websocket.WebSocketApp(ws_url,
                                on_message=on_message,
                                on_error=on_error,
                                on_close=on_close,
                                on_open=on_open)
    ws.run_forever()

# Start WebSocket in background thread
ws_thread = threading.Thread(target=start_websocket)
ws_thread.daemon = True
ws_thread.start()

Connection opened


## 3. Fetch and Parse Order Book Snapshots

Collect and parse incoming WebSocket messages into structured data.

### Fetch Historical Order Book Data (Optional for More Data)

Use CCXT to fetch order book snapshots over a period for backtesting.

In [ ]:
import ccxt

# Fetch historical order book data using CCXT
exchange = ccxt.binance({
    'options': {'defaultType': 'future'},
})

historical_data = []
num_snapshots = 100  # Number of snapshots to fetch

for i in range(num_snapshots):
    try:
        order_book = exchange.fetch_order_book('BTC/USDT', limit=20)
        historical_data.append({
            'timestamp': datetime.now(),
            'bids': order_book['bids'],
            'asks': order_book['asks']
        })
        time.sleep(1)  # Wait 1 second between fetches to avoid rate limits
    except Exception as e:
        print(f"Error fetching data: {e}")
        break

# Convert to DataFrame
df_hist = pd.DataFrame(historical_data)
if not df_hist.empty:
    df_hist['bid_prices'], df_hist['bid_quantities'] = zip(*df_hist['bids'].apply(lambda x: ([p for p, q in x], [q for p, q in x])))
    df_hist['ask_prices'], df_hist['ask_quantities'] = zip(*df_hist['asks'].apply(lambda x: ([p for p, q in x], [q for p, q in x])))
    print(f"Fetched {len(df_hist)} historical snapshots")
else:
    print("No historical data fetched")

# Optionally, combine with real-time data
# df = pd.concat([df, df_hist], ignore_index=True)

In [ ]:
# Collect data for a longer period to get more samples
time.sleep(60)  # Wait for 1 minute of data

with data_lock:
    df = pd.DataFrame(order_book_data)

# Parse bids and asks into separate columns
def parse_levels(levels):
    prices = [float(level[0]) for level in levels]
    quantities = [float(level[1]) for level in levels]
    return prices, quantities

df['bid_prices'], df['bid_quantities'] = zip(*df['bids'].apply(parse_levels))
df['ask_prices'], df['ask_quantities'] = zip(*df['asks'].apply(parse_levels))

print(f"Collected {len(df)} order book snapshots")
print(df.head())

                   timestamp  \
0 2025-11-14 16:08:57.892165   
1 2025-11-14 16:08:58.019279   
2 2025-11-14 16:08:58.212144   
3 2025-11-14 16:08:58.314701   
4 2025-11-14 16:08:58.417370   

                                                bids  \
0  [[96994.20, 2.324], [96994.10, 0.009], [96993....   
1  [[96994.20, 2.316], [96994.10, 0.009], [96993....   
2  [[96994.20, 2.308], [96994.10, 0.017], [96993....   
3  [[96994.20, 2.820], [96994.10, 0.017], [96993....   
4  [[96994.20, 2.818], [96994.10, 0.020], [96993....   

                                                asks   lastUpdateId  \
0  [[96994.30, 1.382], [96994.40, 0.004], [96994....  9190495072070   
1  [[96994.30, 1.382], [96994.40, 0.004], [96994....  9190495078819   
2  [[96994.30, 1.638], [96994.40, 0.004], [96994....  9190495085917   
3  [[96994.30, 0.906], [96994.40, 0.004], [96994....  9190495093752   
4  [[96994.30, 0.911], [96994.40, 0.004], [96994....  9190495103251   

                                          b

## 4. Feature Engineering from Order Book Data

Compute features like bid-ask imbalance, mid-price, spread, and volatility metrics.

In [5]:
# Feature Engineering
def calculate_imbalance(bid_quantities, ask_quantities, k=10):
    bid_sum = sum(bid_quantities[:k])
    ask_sum = sum(ask_quantities[:k])
    return (bid_sum - ask_sum) / (bid_sum + ask_sum) if (bid_sum + ask_sum) > 0 else 0

def calculate_mid_price(bid_prices, ask_prices):
    return (bid_prices[0] + ask_prices[0]) / 2

def calculate_spread(bid_prices, ask_prices):
    return ask_prices[0] - bid_prices[0]

df['imbalance'] = df.apply(lambda row: calculate_imbalance(row['bid_quantities'], row['ask_quantities']), axis=1)
df['mid_price'] = df.apply(lambda row: calculate_mid_price(row['bid_prices'], row['ask_prices']), axis=1)
df['spread'] = df.apply(lambda row: calculate_spread(row['bid_prices'], row['ask_prices']), axis=1)

# Rolling volatility of mid_price
df['mid_price_volatility'] = df['mid_price'].rolling(window=10).std()

# Price change (target for next period)
df['price_change'] = df['mid_price'].shift(-1) - df['mid_price']

print(df[['timestamp', 'imbalance', 'mid_price', 'spread', 'mid_price_volatility', 'price_change']].head())

                   timestamp  imbalance  mid_price  spread  \
0 2025-11-14 16:08:57.892165   0.293715   96994.25     0.1   
1 2025-11-14 16:08:58.019279   0.292330   96994.25     0.1   
2 2025-11-14 16:08:58.212144   0.216409   96994.25     0.1   
3 2025-11-14 16:08:58.314701   0.530211   96994.25     0.1   
4 2025-11-14 16:08:58.417370   0.529042   96994.25     0.1   

   mid_price_volatility  price_change  
0                   NaN           0.0  
1                   NaN           0.0  
2                   NaN           0.0  
3                   NaN           0.0  
4                   NaN           0.0  


## 5. Prepare Dataset for Modeling

Clean the data, create binary labels for price movements, and split into training/validation sets.

In [6]:
# Prepare Dataset
df = df.dropna()  # Drop rows with NaN

# Features
features = ['imbalance', 'spread', 'mid_price_volatility']
X = df[features]

# Target: Binary classification (1 if price increases, 0 otherwise)
y = (df['price_change'] > 0).astype(int)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}, Test set size: {len(X_test)}")

Training set size: 84, Test set size: 22


## 6. Train Supervised Learning Model

Train XGBoost or Random Forest on the engineered features.

In [7]:
# Train XGBoost
model = xgb.XGBClassifier(objective='binary:logistic', n_estimators=100)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

Model Accuracy: 0.91


## 7. Implement Deep Learning Model

Build and train an LSTM model to capture temporal patterns.

### CNN Model for Order Book as Image

Treat the order book as a 2D matrix (bids and asks) and use CNN to extract spatial features.

In [ ]:
# Create 2D representation of order book
def create_order_book_image(bid_quantities, ask_quantities, max_levels=20):
    # Pad or truncate to max_levels
    bids = np.array(bid_quantities[:max_levels])
    asks = np.array(ask_quantities[:max_levels])
    if len(bids) < max_levels:
        bids = np.pad(bids, (0, max_levels - len(bids)), 'constant')
    if len(asks) < max_levels:
        asks = np.pad(asks, (0, max_levels - len(asks)), 'constant')
    # Stack as 2 channels: bids and asks
    image = np.stack([bids, asks], axis=-1)  # Shape: (max_levels, 2)
    return image

# Apply to dataframe
df['order_book_image'] = df.apply(lambda row: create_order_book_image(row['bid_quantities'], row['ask_quantities']), axis=1)

# Prepare data for CNN
X_images = np.stack(df['order_book_image'].values)
X_images = X_images[..., np.newaxis]  # Add channel dimension: (n, 20, 2, 1)
y_images = y.values[:len(X_images)]  # Align with images

X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(X_images, y_images, test_size=0.2, random_state=42)

# Build CNN model
model_cnn = Sequential()
model_cnn.add(Conv2D(32, (3, 1), activation='relu', input_shape=(20, 2, 1)))
model_cnn.add(Flatten())
model_cnn.add(Dense(64, activation='relu'))
model_cnn.add(Dense(1, activation='sigmoid'))

model_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_cnn.fit(X_train_img, y_train_img, epochs=10, batch_size=32, validation_data=(X_test_img, y_test_img))

In [8]:
# Prepare data for LSTM (sequences)
sequence_length = 10
X_seq = []
y_seq = []
for i in range(len(X) - sequence_length):
    X_seq.append(X.iloc[i:i+sequence_length].values)
    y_seq.append(y.iloc[i+sequence_length])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)

# Build LSTM model
model_lstm = Sequential()
model_lstm.add(LSTM(50, input_shape=(sequence_length, len(features))))
model_lstm.add(Dense(1, activation='sigmoid'))

model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_lstm.fit(X_train_seq, y_train_seq, epochs=10, batch_size=32, validation_data=(X_test_seq, y_test_seq))

Epoch 1/10


E0000 00:00:1763107879.960952   78631 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1763107879.966621   78631 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
/home/misango/anaconda3/envs/research_env/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 105ms/step - accuracy: 0.0921 - loss: 0.8548 - val_accuracy: 0.3500 - val_loss: 0.7294
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.3026 - loss: 0.7391 - val_accuracy: 0.8500 - val_loss: 0.6716
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.9342 - loss: 0.6449 - val_accuracy: 0.9000 - val_loss: 0.6249
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9474 - loss: 0.5683 - val_accuracy: 0.9000 - val_loss: 0.5863
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.9474 - loss: 0.5034 - val_accuracy: 0.9000 - val_loss: 0.5533
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9474 - loss: 0.4502 - val_accuracy: 0.9000 - val_loss: 0.5251
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9474 - loss: 0.4012 - val_accuracy: 0.9000 - val_loss: 0.5010
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9474 - loss: 0.3629 - val_accuracy: 0.9000 - val_loss: 0.4801
Epoch 9/10

## 8. Generate Price Predictions

Use the trained models to predict on new data.

In [ ]:
# Predictions with XGBoost
predictions_xgb = model.predict(X_test)

# Predictions with LSTM
predictions_lstm = (model_lstm.predict(X_test_seq) > 0.5).astype(int).flatten()

# Predictions with CNN
predictions_cnn = (model_cnn.predict(X_test_img) > 0.5).astype(int).flatten()

print("XGBoost Predictions:", predictions_xgb[:10])
print("LSTM Predictions:", predictions_lstm[:10])
print("CNN Predictions:", predictions_cnn[:10])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
XGBoost Predictions: [0 1 1 0 0 0 0 0 0 0]
LSTM Predictions: [0 0 0 0 0 0 0 0 0 0]


## 9. Simulate Trading Execution

Implement a simple simulation of trade execution based on predictions and calculate potential profits.

In [10]:
# Simple Trading Simulation
# Assume we trade based on XGBoost predictions
capital = 10000
position = 0  # 1 for long, -1 for short
entry_price = 0

profits = []

for i in range(len(predictions_xgb)):
    pred = predictions_xgb[i]
    actual_change = df.iloc[len(X_train) + i]['price_change']
    current_price = df.iloc[len(X_train) + i]['mid_price']
    
    if position == 0:
        if pred == 1:  # Predict up, go long
            position = 1
            entry_price = current_price
        elif pred == 0:  # Predict down, go short
            position = -1
            entry_price = current_price
    else:
        # Close position
        if position == 1:
            profit = (current_price - entry_price) * 100  # Assume 1 lot
        else:
            profit = (entry_price - current_price) * 100
        profits.append(profit)
        capital += profit
        position = 0

total_profit = sum(profits)
print(f"Total Simulated Profit: {total_profit:.2f}, Final Capital: {capital:.2f}")

Total Simulated Profit: 60.00, Final Capital: 10060.00
